# COMP2050 - Attention U-Net for Medical Image Segmentation
## Notebook 1: Setup & Data Download

In [ ]:
# Install dependencies
!pip install -q segmentation-models-pytorch scipy scikit-learn pandas tqdm

# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Setup Kaggle credentials (using Colab secret)
import json, os
from google.colab import userdata

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

token = userdata.get('KAGGLE_API_TOKEN')
username = 'dngdngphmminh'

creds = {"username": username, "key": token}
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump(creds, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

print('Kaggle credentials configured.')

In [ ]:
# Download Kvasir-SEG dataset from Kaggle
!pip install -q kaggle
!kaggle datasets download -d debeshjha/kvasirseg -p ./data --unzip

# Verify dataset structure
import os
data_dir = './data'
print(f'\nContents of {data_dir}:')
for item in sorted(os.listdir(data_dir)):
    path = os.path.join(data_dir, item)
    if os.path.isdir(path):
        n_files = len(os.listdir(path))
        print(f'  {item}/ ({n_files} files)')
    else:
        print(f'  {item}')

In [ ]:
# Verify dataset loading works
import sys
sys.path.insert(0, '.')

from data.dataset import KvasirSEGDataset
import matplotlib.pyplot as plt
import numpy as np

# Check if images are in a subfolder
import glob
images_in_root = glob.glob('./data/images/*.*')
images_in_sub = glob.glob('./data/Kvasir-SEG/images/*.*')

if images_in_sub and not images_in_root:
    print('Dataset is in ./data/Kvasir-SEG/ subfolder')
elif images_in_root:
    print('Dataset images are directly in ./data/images/')

# Load and visualize a few samples
dataset = KvasirSEGDataset('./data', split='train', seed=42)
print(f'\nTrain set size: {len(dataset)}')

val_dataset = KvasirSEGDataset('./data', split='val', seed=42)
print(f'Val set size: {len(val_dataset)}')

test_dataset = KvasirSEGDataset('./data', split='test', seed=42)
print(f'Test set size: {len(test_dataset)}')

# Visualize 3 samples
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i in range(3):
    img, mask = dataset[i]
    # Denormalize for display
    img_np = img.numpy().transpose(1, 2, 0)
    img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    mask_np = mask.squeeze().numpy()

    axes[0, i].imshow(img_np)
    axes[0, i].set_title(f'Sample {i+1}')
    axes[0, i].axis('off')

    axes[1, i].imshow(mask_np, cmap='gray')
    axes[1, i].set_title(f'Mask {i+1}')
    axes[1, i].axis('off')

plt.suptitle('Kvasir-SEG Dataset Samples')
plt.tight_layout()
plt.show()

In [ ]:
# Upload project code to Colab
# Option 1: If code is on GitHub
# !git clone <your-repo-url> .

# Option 2: Upload files directly
from google.colab import files
print('Upload your project files (models/, losses/, data/, utils/, train.py, etc.)')
print('Or push to GitHub and clone above.')

# Verify all modules import correctly
from models import get_model
from losses import get_loss
from data.dataset import get_dataloaders
from utils.metrics import compute_all_metrics
print('\nAll modules imported successfully!')